# 🏥 TRICARE RAG Q&A 통합 파이프라인

## 개요
- `tricare_rag_pipeline.ipynb`(메인 파이프라인)와 `tricare_exclusion.ipynb`(예외 항목 추가 + 데모 실행)를 하나로 통합한 노트북임
- 단계별로 순서대로 실행하면 전체 RAG Q&A 시스템이 구동됨
- 각 팀원 벡터 DB와 나중에 병합할 수 있도록 경로·컬렉션명을 `tricare_` 프리픽스로 분리 관리함

## 사용 데이터
| 파일 | 형식 | 설명 |
|---|---|---|
| `Costs_Fees.pdf` | PDF | TRICARE 2026 비용·수수료 전체 표 |
| `TOP_Handbook_AUG_2023_FINAL.pdf` | PDF | TRICARE Overseas Program 핸드북 |
| `mental_health_services.csv` | CSV | 정신건강 서비스별 보장 내용 |
| 예외 항목 (크롤러) | 웹 | tricare_exclusions_crawler로 수집한 보장 제외 항목 |

## 사용 모델 / 도구
| 구성 요소 | 선택 |
|---|---|
| LLM | `gpt-4o-mini` |
| 임베딩 | `BAAI/bge-m3` (다국어 100개+) |
| 벡터 DB | Chroma (로컬) |
| PDF 텍스트 | PyMuPDFLoader |
| PDF 표 추출 | pdfplumber |

## 전체 실행 순서
1. 패키지 설치 → 환경 설정 → 파일 경로 설정
2. 임베딩 모델 초기화
3. PDF 로드 → 청킹
4. CSV 로드 → Document 변환 → 청킹
5. 벡터 저장소 생성 (텍스트+CSV 통합 / 표 전용 분리)
6. 예외 항목 크롤링 → 기존 벡터 저장소에 추가
7. LLM + RAG 체인 구성 (다국어 답변 지원)
8. Demo 실행 (일반 LLM 비교 / 텍스트 RAG / 표 RAG)


---
## Section 0 · 패키지 설치

### 개요
- 처음 실행 시에만 필요함
- 주석 해제 후 실행하면 됨


In [1]:
# !pip install langchain langchain-community langchain-chroma langchain-openai \
# pymupdf pdfplumber sentence-transformers chromadb python-dotenv -q


---
## Section 1 · 환경 설정

### 개요
- `.env` 파일에서 `OPENAI_API_KEY`를 읽어옴
- 벡터 DB 경로와 컬렉션명을 상수로 분리해 관리함
- 나중에 팀 통합 벡터 DB를 만들 때 경로만 참고하면 됨

### 벡터 DB 경로 규칙
| 팀원 | persist_directory | collection_name |
|------|------------------|-----------------|
| TRICARE  | `./chroma_db`, `./chroma_db2` | `tricare_rag`, `tricare_cost_tables` |
| Cigna  | `./chroma_cigna_latest`, `./chroma_cigna_all` | `cigna_latest`, `cigna_all` |
| Allianz | `./vectordb` | `allianz_care` |

`.env` 파일 예시:
```
OPENAI_API_KEY=sk-...
```


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

# 벡터 DB 경로·컬렉션명을 상수로 분리
# 나중에 통합 벡터 DB 병합 시 이 상수들을 기준으로 경로를 맞춤
PERSIST_TEXT  = './chroma_db'
PERSIST_TABLE = './chroma_db2'
COLLECTION_TEXT  = 'tricare_rag'
COLLECTION_TABLE = 'tricare_cost_tables'


✅ API Key 로드: 성공


---
## Section 2 · 파일 경로 설정

### 개요
- 세 가지 데이터 소스 경로를 설정함
- 노트북과 같은 디렉터리에 있으면 파일명만 입력해도 됨
- 파일 존재 여부를 ✅/❌로 확인함


In [3]:
PDF_COSTS    = 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf'
PDF_HANDBOOK = 'C:\\SKN_24\\3차 프로젝트\\data\\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf'
CSV_MENTAL   = 'C:\\SKN_24\\3차 프로젝트\\data\\mental_health_services.csv'

for path in [PDF_COSTS, PDF_HANDBOOK, CSV_MENTAL]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'{status} {path}')


✅ C:\SKN_24\3차 프로젝트\data\Costs_Fees.pdf
✅ C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf
✅ C:\SKN_24\3차 프로젝트\data\mental_health_services.csv


---
## Section 3 · 임베딩 모델 초기화 — `BAAI/bge-m3`

### 개요
- 100개 이상의 언어를 동일한 벡터 공간에 매핑하는 다국어 임베딩 모델임
- 한국어 질문과 영어 문서 간 유사도 매칭이 가능함
- 팀원 Cigna 코드와 동일한 모델을 사용해 임베딩 방식을 통일함
- GPU 없는 환경에서는 `device: cpu`로 실행함


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}  # GPU 있으면 'cuda'로 변경
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')


C:\Users\kjw70\AppData\Local\Temp\ipykernel_22296\2816061350.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\kjw70\miniconda3\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kjw70\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ BAAI/bge-m3 임베딩 모델 로드 완료


---
## Section 4 · PDF 로드 — PyMuPDFLoader

### 개요
- `PyPDFLoader` 대신 `PyMuPDFLoader`를 사용함
  - 압축된 최신 PDF에서 파싱 에러가 발생하는 경우가 있음
  - `PyMuPDFLoader`는 동일한 페이지 단위 추출 방식이지만 안정성이 더 높음
- 두 PDF를 로드한 뒤 하나의 리스트로 합쳐서 청킹 진행
- `metadata.source` 필드로 청크 출처 추적 가능함


In [5]:
# !pip install pymupdf


In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

loader_costs    = PyMuPDFLoader(PDF_COSTS)
loader_handbook = PyMuPDFLoader(PDF_HANDBOOK)

docs_costs    = loader_costs.load()
docs_handbook = loader_handbook.load()

# 두 PDF를 하나의 리스트로 합침
all_pdf_docs = docs_costs + docs_handbook

print(f'1. Costs_Fees.pdf: {len(docs_costs)}페이지')
print(f'2. TOP_Handbook.pdf: {len(docs_handbook)}페이지')
print(f'=> 합계: {len(all_pdf_docs)}페이지')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs[0].metadata)


1. Costs_Fees.pdf: 4페이지
2. TOP_Handbook.pdf: 16페이지
=> 합계: 20페이지

--- 샘플 메타데이터 ---
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-28T17:23:05-05:00', 'trapped': '', 'modDate': "D:20260128172305-05'00'", 'creationDate': "D:20260106160556-05'00'", 'page': 0}


---
## Section 5 · PDF 청킹 — RecursiveCharacterTextSplitter

### 개요
- `chunk_size=500`: 보험 약관은 문장이 길고 맥락이 이어지는 경우가 많아 기본값(200)보다 크게 설정함
  - Group A / Group B 분기 설명이 한 단락에 묶여있는 경우가 대표적임
- `chunk_overlap=50`: 청크 경계에서 문맥이 끊기는 것을 방지함
  - 앞 청크의 마지막 50자가 다음 청크 앞부분에 반복되어 맥락이 유지됨
- `separators`: 문단 → 줄바꿈 → 문장 → 공백 → 글자 순으로 우선 분할함
- 이 `text_splitter`는 이후 CSV·예외 항목 청킹에도 재사용됨


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

pdf_chunks = text_splitter.split_documents(all_pdf_docs)

print(f'분할 결과: {len(all_pdf_docs)}페이지 → {len(pdf_chunks)}개 청크')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:300])
print(f'\nMetadata: {pdf_chunks[0].metadata}')


분할 결과: 20페이지 → 107개 청크

--- 청크 샘플 ---
TRICARE® 2026 Costs and Fees
This is an overview of most TRICARE costs and fees. For detailed costs and fees, including those for 
TRICARE For Life, survivors, and medically retired individuals, visit www.tricare.mil/comparecosts. 
ARE YOU IN GROUP A OR GROUP B? 
• You’re in Group A if your initial 

Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-28T17:23:05-05:00', 'trapped': '', 'modDate': "D:20260128172305-05'00'", 'creationDate': "D:20260106160556-05'00'", 'page': 0}


---
## Section 6 · CSV 로드 — Row 단위 Document 변환

### 개요
- `CSVLoader` 대신 직접 변환 방식을 사용함
  - `CSVLoader`는 row 전체를 Document 하나로 만들어 줌
  - `내용` 컬럼이 긴 경우 청킹 시 서비스명(Psychotherapy 등)이 잘려나갈 수 있음
- 직접 변환하면 두 가지를 동시에 보장함
  - `page_content`에 `서비스: ...\n내용: ...` 형식으로 서비스명을 항상 포함함
  - `metadata`에도 서비스명을 저장해 청킹 후에도 출처 추적이 가능함
- 긴 row는 `text_splitter`(Section 5에서 정의)로 추가 청킹함


In [8]:
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv(CSV_MENTAL, encoding='utf-8-sig')

# row 하나를 Document 하나로 변환
# page_content에 서비스명을 직접 포함시켜 청킹 후에도 서비스명이 남도록 함
csv_docs = []
for _, row in df.iterrows():
    doc = Document(
        page_content=f"서비스: {row['서비스']}\n내용: {row['내용']}",
        metadata={
            'source': 'mental_health_services.csv',
            '서비스명': row['서비스'],
            'type': 'mental_health'
        }
    )
    csv_docs.append(doc)

# 500자 넘는 row를 추가 청킹 (Section 5의 text_splitter 재사용)
csv_chunks = text_splitter.split_documents(csv_docs)

print(f'📋 CSV row 수: {len(df)}개')
print(f'🔪 청킹 후 청크 수: {len(csv_chunks)}개')
print(f'\n--- 청크 샘플 ---')
print(csv_chunks[0].page_content[:300])
print(f'\nMetadata: {csv_chunks[0].metadata}')


📋 CSV row 수: 8개
🔪 청킹 후 청크 수: 12개

--- 청크 샘플 ---
서비스: Psychotherapy

Metadata: {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}


In [9]:
# 전체 CSV 청크 메타데이터 확인
# 확인 포인트: 서비스명이 각 청크 metadata에 남아있는지 체크
for i, chunk in enumerate(csv_chunks):
    print(f'[{i}] {chunk.metadata}')


[0] {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
[1] {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
[2] {'source': 'mental_health_services.csv', '서비스명': 'Psychotherapy', 'type': 'mental_health'}
[3] {'source': 'mental_health_services.csv', '서비스명': 'Psychoanalysis', 'type': 'mental_health'}
[4] {'source': 'mental_health_services.csv', '서비스명': 'Psychological testing and assessment', 'type': 'mental_health'}
[5] {'source': 'mental_health_services.csv', '서비스명': 'Psychological testing and assessment', 'type': 'mental_health'}
[6] {'source': 'mental_health_services.csv', '서비스명': 'Psychological testing and assessment', 'type': 'mental_health'}
[7] {'source': 'mental_health_services.csv', '서비스명': 'Psychotropic drugs', 'type': 'mental_health'}
[8] {'source': 'mental_health_services.csv', '서비스명': 'Electroconvulsive therapy', 'type': 'mental_health'}
[9] {'source': 'mental_health_services.csv', '서비스명': 'Coll

---
## Section 7 · 텍스트+CSV 통합 벡터 저장소 생성

### 개요
- PDF 텍스트 청크와 CSV 청크를 하나의 벡터 저장소에 통합함
  - 사용자가 어느 문서에서 답이 나올지 매번 선택하지 않아도 됨
  - Retriever가 소스 구분 없이 가장 관련 있는 청크를 한 번에 찾아줌
- `metadata.source`와 `metadata.type`으로 청크 출처는 항상 추적 가능함
- `persist_directory='./chroma_db'`: 재실행 시 재임베딩 없이 불러올 수 있음


In [10]:
from langchain_chroma import Chroma

all_chunks = pdf_chunks + csv_chunks
print(f'📦 총 청크 수: {len(all_chunks)}개 (PDF: {len(pdf_chunks)} + CSV: {len(csv_chunks)})')
print('⏳ 벡터 저장소 생성 중...')

# PERSIST_TEXT / COLLECTION_TEXT는 Section 1에서 정의한 상수
vector_store = Chroma.from_documents(
    documents=all_chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_TEXT,
    persist_directory=PERSIST_TEXT
)

print(f'✅ Chroma 벡터 저장소 생성 완료')
print(f'   저장된 벡터 수: {vector_store._collection.count()}개')


📦 총 청크 수: 119개 (PDF: 107 + CSV: 12)
⏳ 벡터 저장소 생성 중...
✅ Chroma 벡터 저장소 생성 완료
   저장된 벡터 수: 119개


---
## Section 8 · PDF 표 추출 — pdfplumber

### 개요
- `Costs_Fees.pdf`의 비용 테이블은 일반 텍스트 추출 시 열 정렬이 깨짐
- `pdfplumber`는 표의 셀 좌표를 인식해 `행 | 열` 구조를 보존함
  - 예: `TRICARE Prime | Group A | 외래 진료비 | $26`처럼 정확한 수치 검색이 가능함
- 표는 기존 텍스트 벡터 저장소와 분리해 별도 저장함
  - 비용 질문에 텍스트 청크가 섞이면 노이즈가 될 수 있음
  - 용도에 따라 Retriever를 분리해 검색 정확도를 높임


In [11]:
import pdfplumber

def extract_tables_from_pdf(pdf_path):
    """
    pdfplumber로 PDF 전체에서 표를 추출함.
    각 셀은 ' | '로 구분되어 LLM이 행/열 관계를 파악할 수 있음.
    """
    table_docs = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue
                lines = []
                for row in table:
                    cleaned = [
                        str(cell).strip().replace('\n', ' ') if cell else ''
                        for cell in row
                    ]
                    lines.append(' | '.join(cleaned))
                table_text = '\n'.join(lines)
                table_docs.append(Document(
                    page_content=f'[TRICARE 비용표 | p.{page_num + 1}]\n{table_text}',
                    metadata={
                        'source': pdf_path,
                        'page': page_num + 1,
                        'type': 'cost_table',
                        'table_idx': table_idx + 1
                    }
                ))
    return table_docs

table_docs = extract_tables_from_pdf(PDF_COSTS)
print(f'📊 총 {len(table_docs)}개 표 추출 완료')
print('\n--- 샘플 ---')
print(table_docs[0].page_content[:400])


📊 총 8개 표 추출 완료

--- 샘플 ---
[TRICARE 비용표 | p.1]
ADSMs, ADFMs, and transitional survivors |  | 
Covered service | Group A | Group B
All covered services | $0 | $0
Retirees, their family members, and all others |  | 
Covered service | Group A | Group B
Preventive care visit | $0 | $0
Primary care outpatient visit | $26 | $26
Specialty care outpatient visit | $39 | $39
Urgent care center visit | $39 | $39
Emergency room visit |


---
## Section 9 · 표 전용 벡터 저장소 생성

### 개요
- 비용 수치 질문 전용 벡터 저장소를 별도로 구성함
- `k=3`: 표는 정보 밀도가 높아 3개면 충분함
- `persist_directory='./chroma_db2'`: 텍스트 DB(`./chroma_db`)와 경로를 분리함

### 재실행 시 불러오는 방법
```python
vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)
table_vector_store = Chroma(
    collection_name='tricare_cost_tables',
    embedding_function=embedding_model,
    persist_directory='./chroma_db2'
)
```


In [12]:
# PERSIST_TABLE / COLLECTION_TABLE은 Section 1에서 정의한 상수
table_vector_store = Chroma.from_documents(
    documents=table_docs,
    embedding=embedding_model,
    collection_name=COLLECTION_TABLE,
    persist_directory=PERSIST_TABLE
)

print(f'✅ 표 전용 벡터 저장소: {table_vector_store._collection.count()}개 벡터')


✅ 표 전용 벡터 저장소: 8개 벡터


---
## Section 10 · Retriever 설정

### 개요
- 텍스트+CSV용 `retriever`와 표 전용 `table_retriever`를 각각 설정함
- `k=6`: Group A/B, TRICARE Prime/Select 등 같은 주제가 여러 청크에 분산되어 있어
  k=4면 한쪽 그룹 정보만 검색될 수 있음 → k=6으로 넉넉하게 설정함
- `k=3`: 표 데이터는 밀도가 높아 3개면 충분함


In [13]:
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 6}
)
table_retriever = table_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# 동작 확인
test_result = retriever.invoke('TRICARE Prime 본인부담금')
print(f'🔍 테스트 검색: {len(test_result)}개 문서 검색됨')
for i, d in enumerate(test_result):
    source = d.metadata.get('source', 'unknown')
    print(f'  [{i+1}] {source}: {d.page_content[:80]}...')


🔍 테스트 검색: 6개 문서 검색됨
  [1] C:\SKN_24\3차 프로젝트\data\Costs_Fees.pdf: •	Group B: $462.96 per individual/$927 per family
TRICARE Prime Out-of-Pocket Co...
  [2] C:\SKN_24\3차 프로젝트\data\Costs_Fees.pdf: Group A Retirees and 
their family members
Group B Retirees and their family 
me...
  [3] C:\SKN_24\3차 프로젝트\data\Costs_Fees.pdf: monthly or quarterly premium and follow Group B annual 
deductibles and applicab...
  [4] C:\SKN_24\3차 프로젝트\data\Costs_Fees.pdf: TRICARE Prime Remote, TRICARE Prime Remote 
Overseas, the US Family Health Plan,...
  [5] C:\SKN_24\3차 프로젝트\data\Costs_Fees.pdf: TRICARE Dental Program Monthly Premiums (March 1, 2026–Feb. 28, 2027) 
Sponsor s...
  [6] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf: retain their TRICARE Prime or TRICARE Prime Remote Overseas coverage in the curr...


---
## Section 11 · 예외 항목 추가 — tricare_exclusions_crawler

### 개요
- TRICARE 보장 제외 항목을 크롤러로 수집해 기존 벡터 저장소에 추가함
- 새로운 예외 항목이 생기면 이 셀만 다시 실행하면 됨 (유지보수 포인트)
- `tricare_exclusions_crawler.py`가 노트북과 같은 디렉터리에 있어야 실행 가능함
- 기존 텍스트 벡터 저장소(`tricare_rag`)에 추가하므로 Demo 2에서 자동으로 검색됨


In [14]:
from tricare_exclusions_crawler import crawl_exclusions, to_documents

results = crawl_exclusions()
exclusion_docs = to_documents(results)

# Section 5의 text_splitter 재사용
exclusion_chunks = text_splitter.split_documents(exclusion_docs)
print(f'🔪 제외 항목 청크 수: {len(exclusion_chunks)}개')

# 기존 vector_store에 추가 (재임베딩 없이 기존 저장소에 append)
vector_store.add_documents(exclusion_chunks)
print(f'✅ 추가 후 벡터 수: {vector_store._collection.count()}개')


✅ 제외 항목 링크 55개 수집 완료
  [1/55] Acupuncture 수집 중...
  [2/55] Alterations to Living Space 수집 중...
  [3/55] Alternative Treatments 수집 중...
  [4/55] Assisted Living Facility Care 수집 중...
  [5/55] Augmentation Mammoplasty 수집 중...
  [6/55] Autopsy Services 수집 중...
  [7/55] Aversion Therapy 수집 중...
  [8/55] Camps 수집 중...
  [9/55] Charges for Missed Appointments 수집 중...
  [10/55] Computerized Dynamic Posturography (CDP) 수집 중...
  [11/55] Cosmetic Drugs 수집 중...
  [12/55] Domiciliary Care 수집 중...
  [13/55] Dry Needling 수집 중...
  [14/55] Dynamic Posturography 수집 중...
  [15/55] Dyslexia Treatment 수집 중...
  [16/55] Elective Psychotherapy and Mind Expansion Psychotherapy 수집 중...
  [17/55] Elective Services or Supplies 수집 중...
  [18/55] Elevators or Chair Lifts 수집 중...
  [19/55] Exercise Equipment 수집 중...
  [20/55] Exercise Programs 수집 중...
  [21/55] Experimental Procedures 수집 중...
  [22/55] Fluoride Preparations 수집 중...
  [23/55] Gym Membership 수집 중...
  [24/55] Homeopathic and Herbal Drugs 수집 중...
 

---
## Section 12 · LLM + RAG 체인 구성

### 개요
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
  - `PromptTemplate` 변수로 언어를 넘기면 체인 구조가 복잡해지므로
    f-string으로 프롬프트 문자열에 직접 삽입 후 `PromptTemplate`을 생성함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함
  - 팀원 RAG_TEMPLATE의 `[출처: 문서유형 버전 p.페이지]` 인용 형식과 통일함


In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 언어 감지 함수
# 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리
# 팀원 (rag_utils.py)와 동일한 방식으로 다국어 답변 형식을 통일
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text):
        return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text):
        return 'ko'
    return 'en'

# 언어 코드 → 프롬프트용 언어명 매핑
# 'other'는 LLM이 질문 언어를 그대로 따라가도록 지시
LANGUAGE_NAME_MAP = {
    'ko': 'Korean',
    'en': 'English',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'other': 'the same language as the user\'s question',
}

# format_docs: 출처 정보를 청크 앞에 붙여 반환
# 팀원 RAG_TEMPLATE의 출처 인용 형식([source, p.페이지])과 통일
def format_docs(docs):
    result = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        page   = doc.metadata.get('page', '')
        label  = f'[{source}{f", p.{page+1}" if page else ""}]'
        result.append(f'{label}{doc.page_content}')
    return '\n\n'.join(result)

# make_rag_chain: 질문마다 언어를 감지해 프롬프트에 answer_language를 주입
# retriever를 인자로 받아 텍스트용(retriever)·표 전용(table_retriever) 모두 사용 가능
def make_rag_chain(question: str, retriever, model):
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n\n'
        # 답변 언어 규칙: 감지한 언어를 f-string으로 직접 삽입
        # PromptTemplate 변수로 넘기면 체인 구조가 복잡해지므로 f-string 방식 사용
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- The answer language must match the user\'s question language.\n'
        f'- Do not switch languages unless necessary for official document names or insurance terms.\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )

    rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

    return (
        {'question': RunnablePassthrough(), 'context': retriever | format_docs}
        | rag_prompt
        | model
        | StrOutputParser()
    )

print('✅ LLM + RAG 체인 구성 완료')
print('   호출 방법: make_rag_chain(question, retriever, model).invoke(question)')


✅ LLM + RAG 체인 구성 완료
   호출 방법: make_rag_chain(question, retriever, model).invoke(question)


---
## Section 13 · Demo 1 — 일반 LLM vs RAG 비교

### 개요
- 같은 질문을 일반 LLM과 RAG에 각각 던져 답변 품질을 비교함
- 일반 LLM: 학습 데이터 기반으로 답변 → 세션 종류별 시간 제한 등 세부 조건 누락 가능
- RAG: `mental_health_services.csv`에서 정확한 내용을 검색 후 답변
- 출력 형식: `QUESTION → ANSWER → RETRIEVED DOCS` 순서로 통일함


In [16]:
from langchain_core.messages import HumanMessage

question = (
    'TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? '
    '세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.'
)

print('=' * 100)
print('QUESTION:', question)

print('\n❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답')
print('-' * 65)
plain_response = model.invoke([HumanMessage(content=question)])
print(plain_response.content)
print('\n⚠️  주의: 위 답변은 학습 데이터 기반이며 실제 문서와 다를 수 있습니다.')


QUESTION: TRICARE Prime에서 정신건강 상담(Psychotherapy)은 어떻게 보장되나요? 세션 종류별(개인/가족/그룹) 시간 제한과 주의사항도 알려주세요.

❓ [Demo 1 - 일반 LLM 답변] — 문서 없이 직접 응답
-----------------------------------------------------------------
TRICARE Prime에서 정신건강 상담(Psychotherapy)은 다양한 형태로 제공되며, 보장 내용은 다음과 같습니다:

### 보장 내용
1. **개인 상담**: 개인 심리치료 세션은 일반적으로 보장됩니다. 치료사는 TRICARE 네트워크에 등록된 전문가여야 합니다.
2. **가족 상담**: 가족 단위의 상담도 보장되며, 가족 구성원이 함께 참여하는 세션입니다.
3. **그룹 상담**: 그룹 치료 세션도 포함되며, 여러 사람이 함께 참여하는 형태입니다.

### 세션 시간 제한
- **개인 상담**: 일반적으로 1회 세션은 45분에서 60분 정도입니다.
- **가족 상담**: 가족 상담 세션도 보통 45분에서 60분 정도 소요됩니다.
- **그룹 상담**: 그룹 세션은 보통 1시간에서 2시간 정도 진행됩니다.

### 주의사항
- **사전 승인**: 특정 치료나 세션 수에 따라 사전 승인이 필요할 수 있습니다. 치료를 시작하기 전에 담당 의사와 상담하여 필요한 승인을 받는 것이 중요합니다.
- **네트워크 내 제공자 이용**: TRICARE 네트워크에 등록된 정신건강 전문가를 이용해야 보장이 가능합니다. 네트워크 외의 제공자를 이용할 경우 추가 비용이 발생할 수 있습니다.
- **세션 수 제한**: 특정 진단이나 치료 계획에 따라 보장되는 세션 수에 제한이 있을 수 있습니다. 이는 치료의 필요성에 따라 달라질 수 있습니다.

정확한 정보는 TRICARE 공식 웹사이트나 고객 서비스 센터를 통해 확인하는 것이 좋습니다. 각 개인의 상황에 따라 다를 수 있으므로, 구체적인 사항은 전문가와 상담하는 것

In [17]:
# RAG 체인 생성 (질문 언어 감지 → 답변 언어 자동 설정)
rag_chain = make_rag_chain(question, retriever, model)

retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src = chunk.metadata.get('source', '')
    svc = chunk.metadata.get('서비스명', '')
    print(f'  [{i+1}] {src} {f"({svc})" if svc else ""}: {chunk.page_content[:80]}...')

print()
print('=== ANSWER ===')
rag_response = rag_chain.invoke(question)
print(rag_response)

print('\n=== RETRIEVED DOCS ===')
for doc in retrieved:
    print(doc.metadata)


🔎 검색된 청크:
  [1] mental_health_services.csv (Psychotherapy): 내용: Psychotherapy is discussion-based mental health therapy. It’s also known as ...
  [2] mental_health_services.csv (Psychotherapy): . Family or conjoint sessions: Therapy designed to treat the entire family. Regu...
  [3] https://www.tricare.mil/CoveredServices/IsItCovered/ElectivePsychotherapy : 제외 항목: Elective Psychotherapy and Mind Expansion Psychotherapy
내용: TRICARE doesn...
  [4] mental_health_services.csv (Psychological testing and assessment): 내용: TRICARE covers psychological testing and assessment to help diagnose a patie...
  [5] https://www.tricare.mil/CoveredServices/IsItCovered/MentalHealthExclusions : 제외 항목: Mental Health Exclusions
내용: TRICARE doesn’t cover the following services...
  [6] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf : options TRICARE offers. These options differ in terms of the 
providers you see,...

=== ANSWER ===
TRICARE Prime에서 정신건강 상담(Psychotherapy)은 의학적 또는 심리적으로 필

---
## Section 14 · Demo 2 — 텍스트+CSV+예외 항목 RAG

### 개요
- `question` 변수만 바꿔서 다양한 질문을 테스트할 수 있음
- `retriever`(텍스트+CSV+예외 항목 통합)를 사용함
- 질문 언어를 자동 감지해 같은 언어로 답변함
- 예외 항목은 Section 11에서 벡터 저장소에 추가됐으므로 별도 설정 불필요함

### 사용 방법
- 질문 바꾸고 싶을 때 → `question` 변수만 수정
- Demo 2만 쓰고 싶을 때 → Section 12까지 실행 후 이 셀만 실행
- Demo 2, 3 비교하고 싶을 때 → 두 셀 순서대로 실행


In [18]:
question = '보험 가입 후 취소하고 싶을 때 전액 환불이 가능한 청약 철회 기간(Free Look Period)은 며칠인가요?'
# question = 'I plan on still working when I turn 65. How does this affect my TRICARE?'
# question = 'What is the TRICARE Prime Demo?'

print('=' * 100)
print('QUESTION:', question)

retrieved = retriever.invoke(question)
print('🔎 검색된 청크:')
for i, chunk in enumerate(retrieved):
    src = chunk.metadata.get('source', '')
    print(f'  [{i+1}] {src}: {chunk.page_content[:80]}...')

# 질문 언어 감지 후 체인 생성
rag_chain = make_rag_chain(question, retriever, model)
answer = rag_chain.invoke(question)

print('=== ANSWER ===')
print(answer)

print('\n=== RETRIEVED DOCS ===')
for doc in retrieved:
    print(doc.metadata)


QUESTION: 보험 가입 후 취소하고 싶을 때 전액 환불이 가능한 청약 철회 기간(Free Look Period)은 며칠인가요?
🔎 검색된 청크:
  [1] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf: open enrollment throughout the year.
•	 An initial two-month premium payment is ...
  [2] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf: different TRICARE options 
are available to you. A QLE 
opens a 90-day period fo...
  [3] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf: to see if TRICARE will cover the 
care. To check services that  
need pre-author...
  [4] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf: program managed by Humana Military. CHCBP offers continued health coverage 
(18–...
  [5] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf: that’s above the TRICARE-allowable charge, in addition 
to your deductible and c...
  [6] C:\SKN_24\3차 프로젝트\data\TOP_Handbook_AUG_2023_FINAL_092223_508 (1).pdf: Transitional Assistance Management

---
## Section 15 · Demo 3 — 표 전용 비용 Q&A

### 개요
- `table_retriever`(표 전용)를 사용해 비용 수치를 정확하게 검색함
- 텍스트 청크가 섞이지 않아 `$26`, `$45` 같은 수치 답변이 더 정확함
- 복수 질문을 리스트로 넣어 한 번에 실행 가능함


In [19]:
cost_questions = [
    'TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?',
    'TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?',
    'TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가요?'
]

for q in cost_questions:
    print('=' * 100)
    print('QUESTION:', q)

    # 질문마다 언어 감지 후 체인 생성
    table_rag_chain = make_rag_chain(q, table_retriever, model)
    answer = table_rag_chain.invoke(q)

    print('=== ANSWER ===')
    print(answer)

    print('\n=== RETRIEVED DOCS ===')
    for doc in table_retriever.invoke(q):
        print(doc.metadata)


QUESTION: TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 얼마인가요?
=== ANSWER ===
TRICARE Prime에서 Group A 재향군인 가족의 응급실 방문 비용은 $79입니다.

=== RETRIEVED DOCS ===
{'table_idx': 1, 'type': 'cost_table', 'page': 2, 'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf'}
{'table_idx': 1, 'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'type': 'cost_table', 'page': 1}
{'type': 'cost_table', 'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'table_idx': 1, 'page': 3}
QUESTION: TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 얼마인가요?
=== ANSWER ===
TRICARE Select Group B 재향군인의 네트워크 내 전문의 외래 진료비는 $39입니다.

=== RETRIEVED DOCS ===
{'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'table_idx': 1, 'page': 2, 'type': 'cost_table'}
{'table_idx': 1, 'type': 'cost_table', 'page': 1, 'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf'}
{'type': 'cost_table', 'source': 'C:\\SKN_24\\3차 프로젝트\\data\\Costs_Fees.pdf', 'table_idx': 2, 'page': 1}
QUESTION: TRICARE Reserve Select의 월 보험료(Member only)는 얼마인가

---
## Section 16 · 정리 및 다음 단계

### 데이터 처리 전략 요약
| 데이터 | 처리 방식 | 핵심 이유 |
|---|---|---|
| PDF 텍스트 | PyMuPDFLoader → 청킹 | 안정적 파싱, 페이지 메타데이터 자동 포함 |
| PDF 표 | pdfplumber → Document | 행/열 구조 보존, 수치 정확도 보장 |
| CSV | 직접 row 변환 → 청킹 | 서비스명을 본문+메타데이터 양쪽에 보존 |
| 예외 항목 | 크롤러 → 기존 벡터 저장소 append | 신규 항목 추가 시 해당 셀만 재실행하면 됨 |

### Demo 결과 요약
| | 일반 LLM | RAG 텍스트 | RAG 표 |
|---|---|---|---|
| 데이터 출처 | 학습 데이터 | TRICARE PDF + CSV + 예외 항목 | Costs_Fees.pdf 표 |
| 정확도 | 낮음 | 높음 | 매우 높음 (수치) |
| 다국어 답변 | 가능 | ✅ 언어 감지 + 프롬프트 주입 | ✅ 언어 감지 + 프롬프트 주입 |

### 향후 개선 방향
1. **메타데이터 필터링** — Group A/B, 플랜 종류 필터를 Retriever에 추가
2. **앙상블 Retriever** — BM25(키워드) + bge-m3(의미) 혼합 검색
3. **평가 파이프라인** — RAGAS 라이브러리로 답변 품질 정량 측정
4. **통합 벡터 DB 병합** — 팀원 각자 벡터 DB를 `add_documents()`로 통합 컬렉션에 병합
